# BioPred Phase 0 -- Activity Cleaning and Viability

## Purpose

This notebook loads the raw BioPred GABA-A activity evidence table produced in `02_activity_evidence_ingestion.ipynb` and evaluates whether the extracted data is suitable for downstream modeling.

The goal is not to train models or generate molecular features. The goal is to apply basic usability filters, examine endpoint/value quality, define preliminary cleaning policies, and assess whether the dataset has enough clean molecule-level evidence to justify the next BioPred phase.

## Phase context

This notebook begins the transition from **Phase 0: discovery and data viability assessment** into **Phase 1: curated evidence dataset construction**.

Notebook 02 preserved raw ChEMBL activity evidence at `activity_id` grain. This notebook begins reducing that raw evidence into a cleaner, auditable dataset while preserving endpoint, target, assay, molecule, and source metadata.

## Notebook Objectives

1. Load the raw activity evidence Parquet artifact.
2. Audit row counts, unique molecules, endpoints, units, relations, and missing values before filtering.
3. Apply basic usability filters for molecule structure, endpoint type, units, values, and relation policy.
4. Compute pX values for valid concentration-based activity records.
5. Explore candidate active/inactive thresholds and class prevalence.
6. Audit endpoint and target composition after cleaning.
7. Identify duplicate or repeated molecule-endpoint-target measurements for later aggregation policy.
8. Save a cleaned activity evidence artifact for downstream feature engineering and split planning.

## Design principle

Clean conservatively and audit each reduction step. BioPred should preserve enough metadata to explain which records were removed, which records remained, and why the resulting dataset is or is not viable for ranking-model development.

In [1]:
# Imports to get started
from pathlib import Path
import os

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

# set project root and load the .env file
PROJECT_ROOT = Path.cwd().parent
load_dotenv(PROJECT_ROOT / ".env", override=True)

# create the database engine
db_url = URL.create(
    drivername = os.getenv("BIOPRED_DB_DIALECT"),
    username=os.getenv("BIOPRED_DB_USER"),
    password=os.getenv("BIOPRED_DB_PASSWORD"),
    host=os.getenv("BIOPRED_DB_HOST"),
    port=int(os.getenv("BIOPRED_DB_PORT")),
    database=os.getenv("BIOPRED_DB_NAME")
)

engine = create_engine(db_url, pool_pre_ping=True)
schema = os.getenv("BIOPRED_DB_SCHEMA", "public")

#test connection to db
with engine.connect() as conn:
    result = conn.execute(text("SELECT current_database(), current_schema();"))
    print(result.fetchone())

('chembl36', 'public')


In [5]:
# load our activity_evidence_raw artifact we created from the last notebook
activity_evidence_path = PROJECT_ROOT / "data" / "interim" / "gabaa_activity_evidence_raw.parquet"

activity_evidence_raw = pd.read_parquet(
    PROJECT_ROOT / "data" / "interim" / "gabaa_activity_evidence_raw.parquet"
)


In [4]:
# Verify shape
activity_evidence_raw.shape

# Looking for (5579, 21)

(5579, 21)